# Chapter 2: Being Clear and Direct

**Technique:** Clear and direct instruction

**Contents**
* [Lesson: Explicit Instructions](#lesson)
* [Exercises](#exercises)
* [Example Playground](#example-playground)

In [1]:
# Make the project root importable from any working directory, then load the helpers.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lib" / "helpers.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.helpers import get_completion, response_text, client, MODEL, MAX_TOKENS, SAMPLING_PARAMS

<a id="lesson"></a>
## Lesson: Explicit Instructions

Claude responds best to explicit, direct instructions. When a prompt is vague, Claude fills in the gaps with reasonable defaults, but those defaults rarely match your production requirements.

**State exactly what you want.** Think of a well written Jira ticket or a precise docstring: you'd specify the format, the scope, the expected output, and the audience. The same discipline applies to prompts.

Three dimensions to make explicit:

* **Format**: vague is "tell me about X"; precise is "give me 3 bullet points about X".
* **Length**: vague is "write something about Y"; precise is "write a 400 word design doc about Y".
* **Audience**: vague is "explain Z"; precise is "explain Z to a developer unfamiliar with distributed systems".

A vague prompt forces Claude to guess. A precise prompt is a contract: Claude knows exactly what deliverable to produce. In a production pipeline where output is parsed, logged, or fed into another model, ambiguity is a bug, not a style choice.

**You're not being rude by being direct.** Dropping filler phrases like "could you possibly" and "if you don't mind" does not change how Claude responds to instructions. Claude is not a human colleague; it does not infer goodwill from hedging language. Every token spent on pleasantries is a token that could carry a precise constraint.

In [2]:
# Vague prompt: Claude must guess format, length, and depth
vague_response = get_completion("tell me about errors")
print("=== VAGUE PROMPT OUTPUT ===")
print(vague_response)

# Direct prompt with explicit format, count, and scope
direct_response = get_completion(
    "List 3 common Python asyncio error-handling mistakes as bullet points. "
    "Each bullet: one sentence on what the mistake is, one sentence on how to fix it."
)
print("\n=== DIRECT PROMPT OUTPUT ===")
print(direct_response)

=== VAGUE PROMPT OUTPUT ===
"Errors" covers a lot of ground, so here's an overview of the main areas. Let me know which one you'd like to explore further.

## 1. Programming / Software Errors
- **Syntax errors**: Code violates the language's grammar (e.g., a missing parenthesis). Caught before or at compile time.
- **Runtime errors**: Happen while the program runs (e.g., dividing by zero, accessing a null object, file not found).
- **Logic errors**: The program runs but gives wrong results because the reasoning is flawed. These are often the hardest to find.
- **Error handling**: Mechanisms like `try/catch` (Java, JavaScript), `try/except` (Python), or returned error values (Go, Rust's `Result`) let programs respond gracefully instead of crashing.

## 2. Measurement & Scientific Errors
- **Random error**: Unpredictable fluctuations that scatter results around the true value. Reduced by taking more measurements.
- **Systematic error**: Consistent bias in one direction (e.g., a miscalibr


=== DIRECT PROMPT OUTPUT ===
- **Fire-and-forget tasks:** Calling `asyncio.create_task()` without keeping a reference or awaiting the task means its exceptions go unnoticed (logged only as "Task exception was never retrieved"), and the task may even be garbage-collected mid-execution. Store task references and await them, attach a done callback that checks `task.exception()`, or use `asyncio.TaskGroup` (Python 3.11+) so failures propagate automatically.

- **Misusing `asyncio.gather()`:** With the default `return_exceptions=False`, the first exception propagates to the caller, but the other tasks keep running unsupervised and their later errors are lost. Use `asyncio.TaskGroup`, which cancels sibling tasks on failure, or pass `return_exceptions=True` and explicitly inspect each result for exceptions.

- **Swallowing `CancelledError`:** Using a bare `except:` or `except BaseException:` (or catching `CancelledError` explicitly) without re-raising prevents cancellation and timeouts from 

<a id="exercises"></a>
## Exercises

### Exercise 2.1: Spanish language support response

**Scenario**: your SaaS product serves Spanish speaking users. A customer submits a support ticket asking how to reset their password. You need Claude to answer in Spanish so you can route it directly to the customer without a translation step.

**Task**: Edit `PROMPT` so that Claude's response is in Spanish.

**Success criteria**: the response contains at least one common Spanish word (e.g. `hola`, `gracias`, `buenos`, `cómo`, `ayudar`).

In [3]:
from lib.grading import includes_any, grade

PROMPT = "Eres un agente de soporte. Responde en español, empezando con un saludo. ¿Cómo restablezco mi contraseña?"

response = get_completion(PROMPT)

def grade_exercise(text):
    return includes_any(text, ["hola", "gracias", "buenos", "cómo", "ayudar"])

grade(response, grade_exercise(response))

¡Hola! Gracias por comunicarte con nosotros. Con gusto te ayudo a restablecer tu contraseña. Sigue estos pasos:

1. **Ve a la página de inicio de sesión** y haz clic en **"¿Olvidaste tu contraseña?"**.
2. **Ingresa el correo electrónico** asociado a tu cuenta y haz clic en **"Enviar"**.
3. **Revisa tu bandeja de entrada.** Recibirás un correo con un enlace para restablecer tu contraseña. Si no lo ves en unos minutos, revisa las carpetas de **spam** o **correo no deseado**.
4. **Haz clic en el enlace** del correo. Por seguridad, suele caducar después de un tiempo limitado.
5. **Crea una nueva contraseña** y confírmala. Te recomendamos que:
   - Tenga al menos 8 caracteres.
   - Combine mayúsculas, minúsculas, números y símbolos.
   - No sea una contraseña que hayas usado antes.
6. **Inicia sesión** con tu nueva contraseña.

**Si tienes algún problema:**
- **No recibes el correo:** verifica que escribiste bien la dirección y espera unos minutos antes de solicitarlo de nuevo.
- **El enlac

In [4]:
from hints import exercise_2_1_hint
print(exercise_2_1_hint)

The grader looks for a common Spanish word (e.g. "hola", "gracias"). Tell Claude to reply in Spanish exactly as you'd ask a teammate: directly and explicitly.


### Exercise 2.2: Parseable single word answer

**Scenario**: you're writing a shell script that selects a test framework for a new Python project. You want to call Claude and `grep` the output directly, with no JSON parsing and no stripping prose. The answer must be exactly one word so your script can consume it.

**Task**: Edit `PROMPT` so that Claude replies with ONLY the package name of the most downloaded third-party Python unit testing framework on PyPI, with nothing else: no punctuation, no explanation, no formatting.

**Success criteria**: `response.strip()` equals exactly `"pytest"`.

In [5]:
from lib.grading import equals_exact, grade

PROMPT = "What is the most downloaded third-party Python unit-testing framework on PyPI? Reply with only its package name in lowercase, as a single word, with no punctuation, no quotes, no formatting, and no explanation."

response = get_completion(PROMPT)

def grade_exercise(text):
    return equals_exact(text.strip(), "pytest")

grade(response, grade_exercise(response))

pytest

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


In [6]:
from hints import exercise_2_2_hint
print(exercise_2_2_hint)

The grader checks for the EXACT string "pytest" with no extra words or punctuation. Ask Claude to reply with only the name, in lowercase, and nothing else.


### Exercise 2.3: Full length design document

**Scenario**: your team's RFC process requires design documents of at least 800 words before a proposal can be scheduled for review. A stub or a summary is rejected by the tooling. You need Claude to produce a real draft, not a placeholder.

**Task**: Edit `PROMPT` so that Claude writes a design document of at least 800 words. The topic is up to you, so pick any engineering system (e.g. a rate limiter, a distributed cache, a job queue).

**Success criteria**: the word count of the response is ≥ 800.

In [7]:
from lib.grading import word_count, grade

PROMPT = "Write a detailed software design document of at least 1000 words for a distributed rate limiter. Include these sections, each several paragraphs long: Overview, Goals and Non-Goals, Architecture, Data Model, Algorithms (token bucket and sliding window), Public API, Failure Modes, Scaling, Observability, and Rollout Plan."

response = get_completion(PROMPT)

def grade_exercise(text):
    return word_count(text) >= 800

grade(response, grade_exercise(response))
print("Word count:", word_count(response))

# Design Document: Distributed Rate Limiter

**Status:** Draft for Review
**Component:** Platform Infrastructure / Traffic Management
**Reviewers:** Platform, SRE, API Gateway, Security

---

## 1. Overview

Our services currently enforce rate limits independently, and the implementations vary. Some use in-process token buckets, some rely on API gateway plugins with per-node counters, and a few have no protection at all. Per-instance limiting worked when deployments were small and static. It breaks down under autoscaling. A limit of 100 requests per second per instance becomes 1,000 requests per second when the fleet scales to ten instances, and 3,000 when it scales to thirty. The effective limit therefore depends on fleet size rather than on the business or capacity requirement it was meant to express. Customers see inconsistent behavior, and downstream dependencies are not reliably protected from overload.

This document proposes a **distributed rate limiter**: a shared, low-latency 

In [8]:
from hints import exercise_2_3_hint
print(exercise_2_3_hint)

The grader needs 800+ words. LLMs are poor at counting words, so overshoot your target and ask for a thorough design document.


## Common mistakes

**Asking for quality adjectives instead of measurable constraints**
"Be detailed" and "be thorough" are not instructions, they're hopes. Claude will produce something it considers detailed, which may be 80 words. Give a hard floor: "write at least 800 words" is a constraint; "be thorough" is not.

**Not specifying output language or locale**
If your users are French speaking and your prompt is in English, Claude defaults to English. Specify the output language explicitly: "respond in French" is one phrase that eliminates an entire class of routing bugs.

**Not specifying output format for machine consumption**
If a downstream script parses Claude's output, treat it like an API contract. "Reply with only the status code, no other text" is a format constraint. Without it, Claude may write a sentence explaining the code, and your `int(response)` call raises a `ValueError`.

**Stretch challenge**: Write a prompt that reliably produces exactly a 5 item numbered list, no more and no fewer items. Verify by counting lines in the output (`lib.grading.line_count` does this for you).

**Reflect**: Each retry in a production pipeline costs latency, tokens, and money. How does adding explicit format and length constraints to a prompt reduce the retry rate? What would a prompt quality checklist look like for your team's use case?

<a id="example-playground"></a>
## Example Playground

Use the cell below to experiment freely with direct vs. vague prompts. The graders are not active here.

In [9]:
# Try your own prompt: experiment with explicit format, length, and audience constraints
PROMPT = "In exactly 3 bullet points, list the trade-offs between REST and gRPC for internal microservice communication."
print(get_completion(PROMPT))

- **Performance vs. simplicity:** gRPC uses HTTP/2 and binary Protocol Buffers, giving lower latency, smaller payloads, multiplexed connections, and native bidirectional streaming. That makes it well suited to high-throughput, chatty service-to-service traffic. REST typically uses JSON over HTTP/1.1, which is heavier and slower but simpler to implement and reason about.

- **Strict contracts vs. flexibility:** gRPC requires `.proto` schemas and generates strongly typed client and server code across languages. This catches breaking changes early and enforces consistency, but it adds build-tooling overhead and tighter coupling between services. REST is loosely coupled and schema-optional (unless you add OpenAPI), so it is easier to evolve ad hoc but more prone to runtime mismatches and inconsistent APIs.

- **Tooling and debuggability vs. ecosystem maturity:** REST is human-readable and testable with curl, browsers, or Postman. It works with virtually every proxy, cache, API gateway, and